# Exercício 2 — Duplicatas
**Dataset:** KPMG Customer Transactions (`KPMG_VI_New_raw_data_update_final.xlsx`, planilha *Transactions*)

Objetivo: garantir que cada transação está sendo contada uma única vez antes de calcular faturamento, ticket médio etc.

In [1]:
# IMPORTAÇÕES
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

pd.set_option('display.max_columns', 30)

In [2]:
# CARREGAMENTO DOS DADOS
arquivo = 'KPMG_VI_New_raw_data_update_final.xlsx'
xls = pd.ExcelFile(arquivo)
print('Planilhas:', xls.sheet_names)

# primeira tentativa, do jeito "padrão"
tentativa = pd.read_excel(xls, sheet_name='Transactions')
tentativa.head(3)

Planilhas: ['Title Sheet', 'Transactions', 'NewCustomerList', 'CustomerDemographic', 'CustomerAddress']


,Note: The data and information in this document is reflective of a hypothetical situation and client. This document is to be used for KPMG Virtual Internship purposes only.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
1,1,2,2950,2017-02-25 00:00:00,False,Approved,Solex,Standard,medium,medium,71.49,53.62,41245
2,2,3,3120,2017-05-21 00:00:00,True,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701


## Conferência do cabeçalho e dos tipos

A primeira leitura deu errado: a primeira linha da planilha é uma **nota da KPMG** ("*Note: The data and information in this document is reflective of a hypothetical situation...*"), então o pandas usou essa frase como nome da primeira coluna e o resto virou `Unnamed: 1`, `Unnamed: 2`... Os nomes verdadeiros (`transaction_id`, `product_id`...) ficaram na linha 0 dos dados. Resolvi com `header=1`.

In [3]:
df = pd.read_excel(xls, sheet_name='Transactions', header=1)
print('Dimensões:', df.shape)
df.head()

Dimensões: (20000, 13)


,transaction_id,product_id,customer_id,transaction_date,online_order,order_status,brand,product_line,product_class,product_size,list_price,standard_cost,product_first_sold_date
0,1,2,2950,2017-02-25,0.0,Approved,Solex,Standard,medium,medium,71.49,53.62,41245.0
1,2,3,3120,2017-05-21,1.0,Approved,Trek Bicycles,Standard,medium,large,2091.47,388.92,41701.0
2,3,37,402,2017-10-16,0.0,Approved,OHM Cycles,Standard,low,medium,1793.43,248.82,36361.0
3,4,88,3135,2017-08-31,0.0,Approved,Norco Bicycles,Standard,medium,medium,1198.46,381.10,36145.0
4,5,78,787,2017-10-01,1.0,Approved,Giant Bicycles,Standard,medium,large,1765.30,709.48,42226.0


In [4]:
df.dtypes

transaction_id                      int64
product_id                          int64
customer_id                         int64
transaction_date           datetime64[us]
online_order                      float64
order_status                          str
brand                                 str
product_line                          str
product_class                         str
product_size                          str
list_price                        float64
standard_cost                     float64
product_first_sold_date           float64
dtype: object

In [5]:
# product_first_sold_date veio como número (ex.: 41245). É o número de série de data do Excel.
print(df['product_first_sold_date'].head(3).tolist())
df['product_first_sold_date'] = pd.to_datetime(df['product_first_sold_date'], unit='D', origin='1899-12-30')
print(df['product_first_sold_date'].head(3).tolist())
print()
print('Ausentes por coluna:')
print(df.isna().sum())

[41245.0, 41701.0, 36361.0]
[Timestamp('2012-12-02 00:00:00'), Timestamp('2014-03-03 00:00:00'), Timestamp('1999-07-20 00:00:00')]

Ausentes por coluna:
transaction_id               0
product_id                   0
customer_id                  0
transaction_date             0
online_order               360
order_status                 0
brand                      197
product_line               197
product_class              197
product_size               197
list_price                   0
standard_cost              197
product_first_sold_date    197
dtype: int64


Sobre os tipos:
- `transaction_date` já veio como data.
- `product_first_sold_date` veio como número (41245, 41701...). Isso é o formato interno de data do Excel (dias contados desde 1900), então converti.
- `online_order` veio como `float` só porque tem 360 valores vazios; na prática é um booleano 0/1.
- Tem 197 linhas sem informações do produto (`brand`, `product_line`, etc.). Não é duplicata, mas vou lembrar disso na hora de olhar marcas.

## Duplicatas exatas

In [6]:
# DUPLICATAS EXATAS
print('Linhas completamente iguais:', df.duplicated().sum())

Linhas completamente iguais: 0


Zero. Mas isso já era meio esperado: cada linha tem um `transaction_id` diferente, então mesmo que a mesma compra tenha sido registrada duas vezes, o ID diferente já impede que a linha seja "exatamente igual". Por isso o teste de duplicata exata sozinho não prova muita coisa aqui.

## Duplicatas por subconjunto de colunas

**Quais colunas deveriam identificar uma transação única?** Pensando no negócio (uma loja de bicicletas e acessórios), uma compra é "o mesmo cliente comprando o mesmo produto no mesmo dia, pelo mesmo preço". Então minha chave principal foi:

`customer_id + product_id + transaction_date + list_price`

Deixei o `transaction_id` de fora de propósito, porque é justamente ele que esconderia um registro duplicado (o sistema gera um ID novo a cada inserção). Também testei variações mais frouxas e mais rígidas para ver o que acontece.

In [7]:
# DUPLICATAS POR SUBCONJUNTO DE COLUNAS
chaves = {
    'todas menos transaction_id': [c for c in df.columns if c != 'transaction_id'],
    'cliente + produto + data + preço (minha chave)': ['customer_id', 'product_id', 'transaction_date', 'list_price'],
    'cliente + produto + data': ['customer_id', 'product_id', 'transaction_date'],
    'cliente + data + preço': ['customer_id', 'transaction_date', 'list_price'],
    'todas menos transaction_id e transaction_date': [c for c in df.columns if c not in ('transaction_id', 'transaction_date')],
}
resultado = pd.DataFrame({
    'duplicadas (keep=first)': {k: df.duplicated(subset=v).sum() for k, v in chaves.items()},
    'linhas envolvidas (keep=False)': {k: df.duplicated(subset=v, keep=False).sum() for k, v in chaves.items()},
})
resultado

,duplicadas (keep=first),linhas envolvidas (keep=False)
todas menos transaction_id,0,0
cliente + produto + data + preço (minha chave),0,0
cliente + produto + data,1,2
cliente + data + preço,0,0
todas menos transaction_id e transaction_date,141,281


In [8]:
# o único caso com cliente + produto + data
sub = ['customer_id', 'product_id', 'transaction_date']
df[df.duplicated(subset=sub, keep=False)].T

,10910,16026
transaction_id,10911,16027
product_id,0,0
customer_id,1840,1840
transaction_date,2017-07-11 00:00:00,2017-07-11 00:00:00
online_order,0.0,0.0
order_status,Approved,Approved
brand,OHM Cycles,Solex
product_line,Standard,Standard
product_class,medium,medium
product_size,medium,large


Com a chave "cliente + produto + data" apareceu **um par**. Mas olhando as duas linhas, elas **não são a mesma compra**: uma é uma bicicleta *OHM Cycles* de $235,63 e a outra é uma *Solex* de $202,62. O que elas têm em comum é o `product_id = 0`. Isso me fez desconfiar do próprio `product_id`, então fui investigar os identificadores.

## Unicidade dos identificadores

In [9]:
# UNICIDADE DOS IDENTIFICADORES
print('transaction_id único?', df['transaction_id'].is_unique, '| repetidos:', df['transaction_id'].duplicated().sum())
print('transaction_id vai de', df['transaction_id'].min(), 'a', df['transaction_id'].max(), 'com', df['transaction_id'].nunique(), 'valores')
print()
print('product_id distintos:', df['product_id'].nunique())

# um product_id deveria corresponder a UM produto (mesma marca, linha, classe, tamanho e preço)
atributos = ['brand', 'product_line', 'product_class', 'product_size', 'list_price']
variacao = df.dropna(subset=['brand']).groupby('product_id')[atributos].nunique()
print('product_ids com mais de uma marca:', (variacao['brand'] > 1).sum(), 'de', len(variacao))
print('product_ids com mais de um preço :', (variacao['list_price'] > 1).sum(), 'de', len(variacao))

transaction_id único? True | repetidos: 0
transaction_id vai de 1 a 20000 com 20000 valores

product_id distintos: 101
product_ids com mais de uma marca: 63 de 101
product_ids com mais de um preço : 77 de 101


In [10]:
# exemplo: product_id 0
df[df['product_id'] == 0].groupby(['brand', 'product_line', 'product_size'])['list_price'].agg(['count', 'nunique', 'min', 'max']).head(10)

count  nunique     min     max
brand          product_line product_size                                
Giant Bicycles Standard     large            47        1  569.56  569.56
                            medium           58        1  230.91  230.91
Norco Bicycles Road         medium          114        2  543.39  544.05
               Standard     medium          127        2  360.40  363.01
OHM Cycles     Road         large            50        1   12.01   12.01
                            medium            1        1  742.54  742.54
               Standard     medium          191        4   71.16  235.63
Solex          Road         medium           43        1  416.98  416.98
               Standard     large            27        1  202.62  202.62
                            medium          206        4   71.49  478.16

In [11]:
# quantos "produtos reais" existem se eu definir produto pela combinação de atributos?
produtos_reais = df.dropna(subset=['brand']).drop_duplicates(subset=atributos + ['standard_cost'])
print('Combinações distintas de marca/linha/classe/tamanho/preço/custo:', len(produtos_reais))

Combinações distintas de marca/linha/classe/tamanho/preço/custo: 103


**Conclusão sobre os IDs:**
- `transaction_id` é realmente único: 20.000 valores, de 1 a 20.000, sem repetição nem buraco.
- `product_id` **não identifica um produto**. Dos 101 `product_id`, **63 aparecem com mais de uma marca e 77 com mais de um preço** (o `product_id = 0`, por exemplo, tem bicicletas Giant, Norco, OHM, Solex...). Ao mesmo tempo existem 103 combinações distintas de marca/linha/classe/tamanho/preço/custo, quase o mesmo número de `product_id`, mas elas não batem uma a uma. Minha leitura é que o `product_id` foi gerado de forma aleatória/sintética e não bate com o catálogo.

Na prática isso muda a minha chave de duplicata: em vez de confiar no `product_id`, é mais seguro identificar o produto pelos atributos (`brand`, `product_line`, `product_class`, `product_size`, `list_price`).

In [12]:
# refazendo a checagem com o produto identificado pelos atributos
chave_final = ['customer_id', 'transaction_date'] + atributos
print('Duplicatas pela chave cliente + data + atributos do produto:', df.duplicated(subset=chave_final).sum())

Duplicatas pela chave cliente + data + atributos do produto: 0


In [13]:
# e o caso "tudo igual, menos o ID e a data"?
sem_id_data = [c for c in df.columns if c not in ('transaction_id', 'transaction_date')]
repetidas = df[df.duplicated(subset=sem_id_data, keep=False)].sort_values(['customer_id', 'transaction_date'])
print('Linhas envolvidas:', len(repetidas), '| grupos:', repetidas.groupby(sem_id_data, dropna=False).ngroups)

intervalo = repetidas.groupby(sem_id_data, dropna=False)['transaction_date'].agg(lambda s: (s.max() - s.min()).days)
print()
print('Dias entre a primeira e a última compra do grupo:')
print(intervalo.describe().round(1))
repetidas[['transaction_id', 'customer_id', 'transaction_date', 'brand', 'product_line', 'list_price', 'order_status']].head(6)

Linhas envolvidas: 281 | grupos: 140

Dias entre a primeira e a última compra do grupo:
count    140.0
mean     123.1
std       82.6
min        2.0
25%       57.5
50%      113.5
75%      192.0
max      347.0
Name: transaction_date, dtype: float64


,transaction_id,customer_id,transaction_date,brand,product_line,list_price,order_status
6740,6741,12,2017-02-12,Norco Bicycles,Standard,360.40,Approved
6704,6705,12,2017-09-04,Norco Bicycles,Standard,360.40,Approved
16957,16958,75,2017-06-16,Trek Bicycles,Standard,1057.51,Approved
2483,2484,75,2017-08-22,Trek Bicycles,Standard,1057.51,Approved
19572,19573,139,2017-10-24,Solex,Standard,1163.89,Approved
7226,7227,139,2017-10-31,Solex,Standard,1163.89,Approved


Aqui aparecem 141 "duplicatas" (281 linhas em 140 grupos) se eu ignorar o ID e a data: mesmo cliente, mesmo produto, mesmo preço, mesmo canal. Mas a diferença entre as datas vai de **2 a 347 dias** (mediana ~114 dias). Isso tem muito mais cara de **cliente que comprou o mesmo produto de novo** (uma câmara de ar, uma peça de reposição, um presente) do que de erro de sistema. Se fosse registro duplicado, eu esperaria a mesma data ou uma diferença de minutos/um dia.

## Estratégia de tratamento e impacto

**Decisão:** não removi nenhuma linha.

- Duplicatas exatas: 0.
- Duplicatas pela chave de negócio (cliente + data + produto): 0.
- O único "falso positivo" foi causado pelo `product_id` não confiável.
- As 141 repetições com datas diferentes eu considero compras legítimas.

Para mostrar o impacto da decisão, comparei o indicador "soma de `list_price`" em três cenários: o que eu fiz, o que aconteceria se alguém removesse as "duplicatas" ignorando a data (o erro mais comum), e o que aconteceria se usasse a chave cliente + produto + data sem perceber o problema do `product_id`.

In [14]:
# ESTRATÉGIA DE TRATAMENTO E IMPACTO
df_final = df.copy()   # nenhuma remoção

cenarios = {
    'Minha decisão (nada removido)': df_final,
    'Removendo pela chave cliente+produto+data': df.drop_duplicates(subset=['customer_id', 'product_id', 'transaction_date']),
    'Removendo "tudo igual menos ID e data"': df.drop_duplicates(subset=sem_id_data),
}
impacto = pd.DataFrame({
    nome: {'linhas': len(d),
           'linhas removidas': len(df) - len(d),
           'soma list_price': d['list_price'].sum(),
           'ticket médio': d['list_price'].mean()}
    for nome, d in cenarios.items()
}).T
impacto['diferença na soma'] = impacto['soma list_price'] - df['list_price'].sum()
impacto.style.format({'linhas': '{:,.0f}', 'linhas removidas': '{:,.0f}', 'soma list_price': '$ {:,.2f}',
                      'ticket médio': '$ {:,.2f}', 'diferença na soma': '$ {:,.2f}'})

,linhas,linhas removidas,soma list_price,ticket médio,diferença na soma
Minha decisão (nada removido),"20,000",0,"$ 22,156,588.98","$ 1,107.83",$ 0.00
Removendo pela chave cliente+produto+data,"19,999",1,"$ 22,156,386.36","$ 1,107.87",$ -202.62
"Removendo ""tudo igual menos ID e data""","19,859",141,"$ 22,004,002.34","$ 1,108.01","$ -152,586.64"


In [15]:
# observação extra: "contar cada transação uma única vez" também passa por não contar pedidos cancelados
print(df['order_status'].value_counts())
aprovadas = df[df['order_status'] == 'Approved']
print()
print(f"Soma list_price total     : $ {df['list_price'].sum():,.2f}")
print(f"Soma list_price aprovadas : $ {aprovadas['list_price'].sum():,.2f}")
print(f"Diferença (cancelados)    : $ {df['list_price'].sum() - aprovadas['list_price'].sum():,.2f}")

order_status
Approved     19821
Cancelled      179
Name: count, dtype: int64

Soma list_price total     : $ 22,156,588.98
Soma list_price aprovadas : $ 21,954,196.69
Diferença (cancelados)    : $ 202,392.29


**Resumo do impacto:**
- Registros afetados pela minha estratégia: **0**. A soma de `list_price` continua em ~$22,16 milhões.
- Se eu tivesse removido as 141 linhas "repetidas" sem olhar a data, teria jogado fora cerca de **$152,6 mil** de vendas reais (tabela acima) e ainda subestimado a recorrência de clientes, que é justamente uma informação valiosa para marketing.
- Não é duplicata, mas vale registrar: 179 pedidos estão **cancelados**. Para calcular faturamento eles deveriam sair da conta, senão também estou contando algo que não aconteceu.

## Respostas às perguntas de reflexão

**1. Se o resultado de duplicatas exatas for zero, isso garante que não há registros repetidos? Por quê?**

Não garante. Foi exatamente o caso aqui: qualquer coluna que muda a cada inserção (um ID automático, um timestamp de carga, um espaço a mais no nome) já faz duas linhas que representam a mesma coisa deixarem de ser "exatamente iguais". O `duplicated()` compara tudo, caractere por caractere. Por isso o teste de duplicata exata é só o primeiro passo; o que realmente importa é definir quais colunas, juntas, representam "a mesma transação" e testar por elas. E mesmo assim podem existir duplicatas "quase iguais" (como "Giant Bicycles" e "Giant Bicycle"), que só aparecem com comparação aproximada.

**2. Em uma base de transações, duas linhas idênticas são sempre um erro? Dê um exemplo em que não seriam.**

Não. Se a base não tiver um ID por transação (ou se o nível de detalhe for "item do pedido"), duas linhas iguais podem ser legítimas. Exemplo: um pai compra duas bicicletas iguais para os dois filhos no mesmo dia, na mesma loja. Se o sistema registra um item por linha sem coluna de quantidade, vão aparecer duas linhas idênticas e as duas são vendas reais. Outro exemplo é um cliente que passa duas vezes no caixa no mesmo dia comprando o mesmo energético. Remover uma delas seria apagar uma venda que aconteceu.

**3. Por que a escolha das colunas que definem "o mesmo registro" é uma decisão de negócio, e não apenas técnica? Relacione com o Entendimento do Negócio do CRISP-DM.**

Porque o código (`drop_duplicates(subset=...)`) é igual para qualquer escolha; o que muda é o significado. Aqui eu tive que decidir se duas compras do mesmo produto pelo mesmo cliente com 2 dias de diferença são uma recompra ou um erro, e isso só dá para responder sabendo como a empresa funciona: ela vende peças de reposição? Existe processo de troca que gera um novo registro? O sistema de vendas costuma duplicar pedidos? Também descobri que o `product_id` não identifica o produto, algo que eu teria confirmado com quem conhece o sistema antes de escolher a chave.

Na fase de **Entendimento do Negócio** do CRISP-DM é que se define o objetivo da análise e se conversa com quem conhece o processo. É lá que deveria sair a definição de "o que é uma transação" e "o que é um cliente". Se eu pulo essa etapa e escolho as colunas só pelo que parece tecnicamente razoável, posso tanto apagar vendas reais quanto manter registros duplicados, e os dois erros distorcem o faturamento.

## Desafio 1 — Duplicatas aproximadas com `rapidfuzz`

A ideia é comparar cada valor de texto com os outros e ver quais são muito parecidos (score de 0 a 100). Testei primeiro na coluna `brand` e depois em colunas mais "sujas" das outras planilhas.

In [16]:
def pares_parecidos(valores, limite=85, scorer=fuzz.token_sort_ratio):
    valores = [v for v in pd.Series(valores).dropna().astype(str).unique()]
    pares = []
    for i, a in enumerate(valores):
        for b, score, _ in process.extract(a, valores[i + 1:], scorer=scorer, limit=None):
            if score >= limite:
                pares.append((a, b, round(score, 1)))
    return pd.DataFrame(pares, columns=['valor_a', 'valor_b', 'score']).sort_values('score', ascending=False)

print('Marcas distintas:', df['brand'].dropna().unique().tolist())
print()
# com limite baixo, para ver o que o algoritmo considera "parecido"
pares_parecidos(df['brand'], limite=60, scorer=fuzz.ratio)

Marcas distintas: ['Solex', 'Trek Bicycles', 'OHM Cycles', 'Norco Bicycles', 'Giant Bicycles', 'WeareA2B']



,valor_a,valor_b,score
0,Trek Bicycles,Norco Bicycles,74.1
1,Trek Bicycles,Giant Bicycles,66.7
2,Norco Bicycles,Giant Bicycles,64.3


In [17]:
# teste de sanidade com o exemplo do enunciado
print('Giant Bicycles x Giant Bicycle :', fuzz.ratio('Giant Bicycles', 'Giant Bicycle'))
print('Giant Bicycles x Trek Bicycles :', fuzz.ratio('Giant Bicycles', 'Trek Bicycles'))
print('Giant Bicycles x giant bicycles:', fuzz.ratio('Giant Bicycles', 'giant bicycles'),
      '| com processamento:', fuzz.ratio('Giant Bicycles', 'giant bicycles', processor=str.lower))

Giant Bicycles x Giant Bicycle : 96.2962962962963
Giant Bicycles x Trek Bicycles : 66.66666666666667
Giant Bicycles x giant bicycles: 85.71428571428572 | com processamento: 100.0


A coluna `brand` desta base já está limpa (6 marcas, sem variações). O interessante é que, com um limite baixo, o `rapidfuzz` aponta "Giant Bicycles" e "Trek Bicycles" como parecidas só por causa da palavra "Bicycles", e elas são marcas completamente diferentes. Ou seja: **o score sozinho não decide nada**, alguém precisa olhar os pares e escolher o limite. O exemplo do enunciado ("Giant Bicycles" x "Giant Bicycle") dá 96, bem acima de "Giant" x "Trek".

Fui procurar um caso real de variação nas outras planilhas. Na *CustomerAddress*, a coluna `state` tem esse problema:

In [18]:
enderecos = pd.read_excel(xls, sheet_name='CustomerAddress', header=1)
print(enderecos['state'].value_counts())

state
NSW                2054
VIC                 939
QLD                 838
New South Wales      86
Victoria             82
Name: count, dtype: int64


In [19]:
# o fuzzy padrão não pega abreviação x nome completo...
for a, b in [('NSW', 'New South Wales'), ('VIC', 'Victoria'), ('QLD', 'Queensland')]:
    print(f'{a} x {b}: ratio={fuzz.ratio(a, b):.0f} | partial_ratio={fuzz.partial_ratio(a.lower(), b.lower()):.0f}')

# ...então comparo a sigla com as iniciais/consoantes do nome completo
def sigla(nome):
    palavras = nome.split()
    return ''.join(p[0] for p in palavras).upper() if len(palavras) > 1 else nome[:3].upper()

print()
estados = enderecos['state'].unique()
for completo in [e for e in estados if len(e) > 3]:
    melhor = process.extractOne(sigla(completo), [e for e in estados if len(e) <= 3], scorer=fuzz.ratio)
    print(f'{completo!r:20} -> sigla {sigla(completo)!r:6} -> casa com {melhor[0]!r} (score {melhor[1]:.0f})')

NSW x New South Wales: ratio=33 | partial_ratio=67
VIC x Victoria: ratio=18 | partial_ratio=100
QLD x Queensland: ratio=15 | partial_ratio=50

'New South Wales'    -> sigla 'NSW'  -> casa com 'NSW' (score 100)
'Victoria'           -> sigla 'VIC'  -> casa com 'VIC' (score 100)


In [20]:
# e em job_title (planilha CustomerDemographic), que tem muito mais valores
demo = pd.read_excel(xls, sheet_name='CustomerDemographic')
print('Cargos distintos:', demo['job_title'].nunique())
pares_parecidos(demo['job_title'], limite=88).head(15)

Cargos distintos: 195


,valor_a,valor_b,score
22,Budget/Accounting Analyst I,Budget/Accounting Analyst II,98.2
23,Budget/Accounting Analyst I,Budget/Accounting Analyst IV,98.2
92,Budget/Accounting Analyst II,Budget/Accounting Analyst III,98.2
131,Computer Systems Analyst II,Computer Systems Analyst III,98.2
125,Administrative Assistant III,Administrative Assistant II,98.2
111,Human Resources Assistant II,Human Resources Assistant III,98.2
56,Human Resources Assistant I,Human Resources Assistant II,98.2
57,Human Resources Assistant I,Human Resources Assistant IV,98.2
16,Computer Systems Analyst I,Computer Systems Analyst II,98.1
97,Administrative Assistant IV,Administrative Assistant I,98.1


Nas siglas de estado, o `ratio` puro dá score baixo (NSW x New South Wales não tem quase nenhuma letra em comum na mesma ordem). Precisei de uma regrinha (gerar a sigla do nome completo) antes de comparar. Isso mostra que fuzzy matching ajuda, mas não substitui conhecer o domínio: eu sei que NSW é New South Wales, o algoritmo não sabe.

Já em `job_title`, os pares com score alto são quase todos cargos **diferentes** de verdade, como "Budget/Accounting Analyst I" x "Budget/Accounting Analyst II" ou "Administrative Assistant III" x "Administrative Assistant II" (score 98!). Juntar esses seria um erro grave. De novo: o fuzzy serve para levantar suspeitos, não para decidir sozinho.

## Desafio 2 — Gênero na planilha CustomerDemographic

In [21]:
print(demo['gender'].value_counts(dropna=False))
print()
print('Valores distintos (exatos, com repr para ver espaços):', [repr(v) for v in demo['gender'].unique()])

gender
Female    2039
Male      1873
U           88
Name: count, dtype: int64

Valores distintos (exatos, com repr para ver espaços): ["'U'", "'Male'", "'Female'"]


In [22]:
# padronização: deixo explícito o mapeamento e transformo 'U' em ausente
mapa_genero = {'female': 'Female', 'f': 'Female', 'femal': 'Female',
               'male': 'Male', 'm': 'Male',
               'u': np.nan, 'unknown': np.nan}
demo['gender_limpo'] = demo['gender'].astype(str).str.strip().str.lower().map(mapa_genero)
print(demo['gender_limpo'].value_counts(dropna=False))

gender_limpo
Female    2039
Male      1873
NaN         88
Name: count, dtype: int64


Nesta versão do arquivo a coluna `gender` tem **3 formas de registro**: `Female`, `Male` e `U`. As versões mais antigas desse dataset que circulam no Kaggle também traziam `F`, `Femal` e `M`. Aqui elas já não aparecem, mas deixei o mapeamento preparado para esses casos (normalizando para minúsculas e sem espaços antes de mapear).

O ponto que achei mais importante é o `U`: são **88 clientes** com gênero "desconhecido" (*unknown*/*unspecified*). Para o pandas isso é uma categoria válida, mas na prática é um valor ausente disfarçado. Se alguém fizer um gráfico de vendas por gênero sem tratar isso, vai aparecer um terceiro "gênero" que na verdade é falta de informação. Por isso converti `U` para `NaN`.